In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langgraph.graph import MessagesState
from typing_extensions import Annotated
from operator import add

class State(MessagesState):
    retrieved_doc: Annotated[list[dict], add]

In [3]:
import os
from tavily import TavilyClient

def search_node(state: State):    
    client = TavilyClient(os.getenv("TAVILY_KEY"))
    response = client.search(
        query=str(state['messages'][-1].content),
        search_depth="advanced",
        include_usage=True,
        #include_raw_content="markdown"
    )
    return {"retrieved_doc": response['results']}

    

In [4]:
from langchain.chat_models import init_chat_model
from coursegen.prompts.answer import ANSWER_PROMPT

def answer_node(state: State):
    model = init_chat_model("gpt-4o-mini")
    response = model.invoke(ANSWER_PROMPT.format(question=state['messages'][-1].content, retrieved_doc=state['retrieved_doc']))
    return {'messages':[response]}

/Users/ken/Documents/CS/CourseGen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain.messages import AIMessage
from coursegen.prompts.examine import EXAMINE_FEYNMAN_PROMPT

def examine_node(state: State):
    model = init_chat_model("gpt-4o-mini")
    response = model.invoke(state['messages'] + [AIMessage(EXAMINE_FEYNMAN_PROMPT)])
    return {'messages':[response]}

In [6]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("search_node", search_node)
builder.add_node("answer_node", answer_node)
builder.add_node("examine_node", examine_node)
builder.add_edge(START, "search_node")
builder.add_edge("search_node", "answer_node")
builder.add_edge("answer_node", "examine_node")
builder.add_edge("examine_node", END)
graph = builder.compile()

In [7]:
from langchain.messages import HumanMessage

result = graph.invoke({"messages": [HumanMessage("How to defeat Warden?")]})
result

{'messages': [HumanMessage(content='How to defeat Warden?', additional_kwargs={}, response_metadata={}, id='aa770e88-e904-494d-813d-d83702047845'),
  AIMessage(content='### 學習路線圖：如何擊敗 Warden\n\n#### 1. 理解 Warden 的特性\n- **位置和環境**：Warden 存在於深暗生物群落中，依賴振動、氣味和觸覺進行偵測。因此，靜音行動是避開其注意的好方法。\n- **攻擊方式**：\n  - **近戰攻擊**：每次擊中能造成高達 22.5 顆心的傷害（依難度而異）。\n  - **音爆攻擊**：可以穿透障礙物，與目標相距 20 方塊內時發動。\n  - **視覺特徵**：Warden 是盲目的，使用夜視藥水可以稍微抵消黑暗效果。\n\n#### 2. 準備對戰裝備\n- **護甲**：\n  - **必備**：全套下界合金護甲，並附上最高等級的“保護”魔法。\n- **武器**：\n  - **遠程武器**：使用弓加上“力量 V”的箭或慢速 IV 箭來擊敗 Warden。\n  - **輔助武器**：可以攜帶劍作為近戰防備，強烈建議以遠程方式作戰。\n  \n#### 3. 戰鬥策略\n- **引誘 Warden**：\n  - 在 Sculk Shrieker 附近造成噪音，引誘 Warden 來到預定地點。\n- **利用地形**：\n  - 找一個有足夠垂直空間的大洞穴，並建立高台（至少 20 方塊高），這樣 Warden 無法使用音爆攻擊。\n  \n#### 4. 實施對抗策略\n- **使用物品分散注意力**：\n  - 丟擲蛋或其他物品來分散 Warden 的注意力，避免直接攻擊。\n- **Potion 使用**：\n  - 使用提升速度的藥水或減弱 Warden 的效果藥水（如弱化藥水）。\n- **交替攻擊**：\n  - 確保在進行近戰攻擊後，立刻拉開距離並使用遠程武器，不斷射擊 Warden。\n  \n#### 5. 重複策略\n- **禁錮與反擊**：\n  - 當 Warden 繼續接近時，利用策略重整位置，並持續射擊，直到擊敗對手。\n